# Otomotiv Kredi Finansmanı — PD / LGD / EAD / Expected Loss Demo

Bu notebook, Türkiye otomotiv kredi finansmanı (dealer bazlı araç kredisi) bağlamına
uyarlanmış, sentetik veriyle çalışan bir **kredi risk modelleme** demosudur.

Amaç: gerçek dealer/kredi veri şemana benzer bir yapı üzerinden,
**PD (temerrüt olasılığı) → LGD (temerrütte kayıp oranı) → EAD (temerrüt anındaki risk tutarı)**
zincirini kurup, portföy bazında **Expected Loss (EL)** hesaplamak ve
**what-if senaryoları** çalıştırabilecek bir simülatör fonksiyonu üretmek.

> Not: Veri tamamen sentetiktir, gerçek dealer/müşteri verisi içermez.
> Amaç metodolojiyi göstermek; production'da kendi Oracle DB / FastAPI pipeline'ına bağlanacak.


## 1. Kurulum ve sentetik veri üretimi

Gerçek otomotiv kredi finansman şemana benzer alanlar kullanıyoruz: dealer, araç segmenti, müşteri profili, Findeks/KKB skoru, kredi koşulları ve temerrüt/tahsilat bilgisi.

In [9]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

np.random.seed(42)
N = 8000  # başvuru sayısı

dealers = [f"DLR-{i:03d}" for i in range(1, 41)]
regions = ["Marmara", "Ege", "İç Anadolu", "Akdeniz", "Karadeniz", "Doğu Anadolu", "Güneydoğu Anadolu"]
vehicle_segments = ["Binek - Ekonomi", "Binek - Orta", "SUV", "Hafif Ticari"]
brand_tiers = ["Yerli", "İthal - Ekonomik", "İthal - Premium"]
employment_types = ["Ücretli (SGK'lı)", "Serbest Meslek", "Esnaf/KOBİ", "Emekli"]

df = pd.DataFrame({
    "application_id": [f"APP-{i:06d}" for i in range(N)],
    "dealer_id": np.random.choice(dealers, N),
    "region": np.random.choice(regions, N, p=[.30,.12,.18,.13,.10,.07,.10]),
    "vehicle_segment": np.random.choice(vehicle_segments, N, p=[.35,.30,.20,.15]),
    "brand_tier": np.random.choice(brand_tiers, N, p=[.45,.35,.20]),
    "vehicle_is_used": np.random.choice([0,1], N, p=[.55,.45]),  # 0=sıfır, 1=2.el
    "customer_age": np.clip(np.random.normal(38, 11, N), 19, 75).astype(int),
    "customer_income_tl": np.clip(np.random.lognormal(10.4, 0.45, N), 18000, 350000).round(-2),
    "employment_type": np.random.choice(employment_types, N, p=[.55,.18,.17,.10]),
    "findeks_score": np.clip(np.random.normal(1450, 220, N), 300, 1900).astype(int),
    "existing_loan_count": np.random.poisson(1.1, N),
})

# Kredi koşulları
df["loan_amount_tl"] = (df["customer_income_tl"] * np.random.uniform(2.5, 9, N)).round(-2)
df["down_payment_ratio"] = np.clip(np.random.beta(2.2, 4.5, N), 0.05, 0.75).round(3)
df["term_months"] = np.random.choice([12,24,36,48,60], N, p=[.05,.20,.30,.30,.15])
df["financed_amount_tl"] = (df["loan_amount_tl"] * (1 - df["down_payment_ratio"])).round(-1)
df["monthly_installment_tl"] = (df["financed_amount_tl"] * 1.12 / df["term_months"]).round(-1)  # basit faiz yaklaşımı
df["installment_to_income"] = (df["monthly_installment_tl"] / (df["customer_income_tl"]/1)).round(3)

df.head()


,application_id,dealer_id,region,vehicle_segment,brand_tier,vehicle_is_used,customer_age,customer_income_tl,employment_type,findeks_score,existing_loan_count,loan_amount_tl,down_payment_ratio,term_months,financed_amount_tl,monthly_installment_tl,installment_to_income
0,APP-000000,DLR-039,Güneydoğu Anadolu,Hafif Ticari,Yerli,1,35,57400.0,Emekli,1278,0,310900.0,0.079,48,286340.0,6680.0,0.116
1,APP-000001,DLR-029,İç Anadolu,Binek - Orta,İthal - Premium,0,48,22000.0,Ücretli (SGK'lı),1413,2,98000.0,0.137,48,84570.0,1970.0,0.090
2,APP-000002,DLR-015,Marmara,SUV,İthal - Ekonomik,0,28,22100.0,Ücretli (SGK'lı),1802,0,79800.0,0.290,60,56660.0,1060.0,0.048
3,APP-000003,DLR-008,Marmara,Binek - Ekonomi,Yerli,1,34,18000.0,Ücretli (SGK'lı),1622,2,77300.0,0.341,24,50940.0,2380.0,0.132
4,APP-000004,DLR-021,İç Anadolu,Binek - Orta,Yerli,1,32,49700.0,Ücretli (SGK'lı),1383,0,395700.0,0.111,48,351780.0,8210.0,0.165


## 2. Default (temerrüt) ve FPD etiketinin üretilmesi

Gerçek hayattaki bilinen risk sürücülerini (düşük Findeks skoru, yüksek taksit/gelir oranı, düşük peşinat, esnaf/KOBİ segmenti, 2.el araç) kullanarak temerrüt olasılığını simüle ediyoruz — yani 'doğru cevabı' biliyoruz, modelin bunu ne kadar yakaladığını test edeceğiz.

In [10]:

# Gerçekçi bir temerrüt mantığı kuruyoruz (logit tabanlı simülasyon)
logit = (
    -3.2
    - 0.0032 * (df["findeks_score"] - 1450)          # yüksek skor -> düşük risk
    + 3.1 * df["installment_to_income"]                # yüksek taksit/gelir -> yüksek risk
    - 1.8 * df["down_payment_ratio"]                   # yüksek peşinat -> düşük risk
    + 0.35 * df["existing_loan_count"]
    + 0.55 * (df["employment_type"] == "Esnaf/KOBİ").astype(int)
    + 0.30 * df["vehicle_is_used"]
    + np.random.normal(0, 0.55, N)                     # gürültü
)
prob_default = 1 / (1 + np.exp(-logit))
df["default_flag"] = (np.random.rand(N) < prob_default).astype(int)

# FPD (First Payment Default): default edenlerin bir kısmı ilk taksitte düşer
df["is_fpd"] = 0
default_idx = df[df["default_flag"]==1].index
fpd_mask = np.random.rand(len(default_idx)) < 0.22  # default edenlerin ~%22si FPD
df.loc[default_idx[fpd_mask], "is_fpd"] = 1

print(f"Toplam başvuru: {N}")
print(f"Default oranı: {df['default_flag'].mean():.2%}")
print(f"Default içinde FPD oranı: {df['is_fpd'].sum() / df['default_flag'].sum():.2%}")


Toplam başvuru: 8000
Default oranı: 9.04%
Default içinde FPD oranı: 20.47%


## 3. LGD ve EAD için tahsilat/bakiye verisi (sadece default edenler için)

LGD ve EAD, yalnızca temerrüde düşmüş krediler için anlamlıdır — bu yüzden bu alanlar default=1 olan satırlarda dolduruluyor (gerçek dünyada da bu böyledir).

In [11]:

df["months_survived"] = np.where(
    df["default_flag"]==1,
    np.where(df["is_fpd"]==1, 1, np.random.randint(2, np.maximum(df["term_months"]-2,3))),
    df["term_months"]
)

# Temerrüt anındaki kalan bakiye (basit amortisman yaklaşımı)
remaining_ratio = 1 - (df["months_survived"] / df["term_months"])
df["outstanding_at_default_tl"] = np.where(
    df["default_flag"]==1,
    (df["financed_amount_tl"] * remaining_ratio).round(-1),
    np.nan
)

# Teminat (araç) değeri ve tahsilat — 2.el/yerli araçlarda likidite daha yüksek -> recovery daha iyi
collateral_recovery_rate = np.where(
    df["brand_tier"]=="İthal - Premium", np.random.beta(5,3,N),
    np.where(df["brand_tier"]=="Yerli", np.random.beta(6,2.5,N), np.random.beta(5,3,N))
)
noise = np.random.normal(0, 0.08, N)
df["recovery_rate"] = np.where(
    df["default_flag"]==1,
    np.clip(collateral_recovery_rate + noise, 0.05, 0.98),
    np.nan
)
df["recovered_amount_tl"] = (df["outstanding_at_default_tl"] * df["recovery_rate"]).round(-1)
df["lgd"] = 1 - df["recovery_rate"]

df[df["default_flag"]==1][["application_id","brand_tier","outstanding_at_default_tl","recovery_rate","lgd"]].head()


,application_id,brand_tier,outstanding_at_default_tl,recovery_rate,lgd
36,APP-000036,Yerli,220180.0,0.768816,0.231184
48,APP-000048,Yerli,201500.0,0.519445,0.480555
62,APP-000062,İthal - Ekonomik,15740.0,0.880213,0.119787
87,APP-000087,İthal - Ekonomik,19830.0,0.668523,0.331477
100,APP-000100,İthal - Ekonomik,9750.0,0.721202,0.278798


## 4. PD Modeli — Lojistik Regresyon (Scorecard mantığı)

Gerçek scorecard projelerinde WoE/IV binning yapılır; burada demo amaçlı sadeleştirip doğrudan sayısal ve kategorik değişkenlerle lojistik regresyon kuruyoruz. Aşama, mantığı göstermek — production'da WoE binning + IV seçimi eklenmeli.

In [12]:

features_num = ["findeks_score","installment_to_income","down_payment_ratio",
                "existing_loan_count","customer_age"]
features_cat = ["employment_type","vehicle_segment","brand_tier","region"]

X = pd.get_dummies(df[features_num + features_cat], columns=features_cat, drop_first=True)
y = df["default_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pd_model = LogisticRegression(max_iter=2000, class_weight="balanced")
pd_model.fit(X_train, y_train)

y_proba = pd_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_proba)
print(f"PD Modeli — Test AUC: {auc:.3f}")

# Katsayılara bakarak en etkili faktörler (basitleştirilmiş scorecard mantığı)
coef_df = pd.DataFrame({"feature": X.columns, "coef": pd_model.coef_[0]}).sort_values("coef")
print("\nEn çok riski ARTIRAN 5 faktör:")
print(coef_df.tail(5).to_string(index=False))
print("\nEn çok riski AZALTAN 5 faktör:")
print(coef_df.head(5).to_string(index=False))


PD Modeli — Test AUC: 0.751

En çok riski ARTIRAN 5 faktör:
                   feature     coef
  region_Güneydoğu Anadolu 0.260129
       existing_loan_count 0.314411
       region_Doğu Anadolu 0.371437
employment_type_Esnaf/KOBİ 0.569888
     installment_to_income 2.391024

En çok riski AZALTAN 5 faktör:
                     feature      coef
          down_payment_ratio -1.582697
vehicle_segment_Hafif Ticari -0.217033
                  region_Ege -0.129639
            region_Karadeniz -0.040272
               findeks_score -0.003362


## 5. LGD Modeli

Sadece default eden krediler üzerinde, recovery_rate'i (1 - LGD) tahmin eden bir regresyon kuruyoruz. Ana sürücü: teminat (araç) tipi ve marka kademesi.

In [13]:

from sklearn.linear_model import LinearRegression

df_def = df[df["default_flag"]==1].copy()
X_lgd = pd.get_dummies(df_def[["brand_tier","vehicle_is_used","vehicle_segment"]], drop_first=True)
y_lgd = df_def["recovery_rate"]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(X_lgd, y_lgd, test_size=0.25, random_state=42)
lgd_model = LinearRegression()
lgd_model.fit(Xl_train, yl_train)

from sklearn.metrics import r2_score
pred_recovery = lgd_model.predict(Xl_test)
print(f"LGD Modeli (recovery_rate tahmini) — Test R²: {r2_score(yl_test, pred_recovery):.3f}")
print(f"Ortalama gerçek recovery rate: {yl_test.mean():.2%} | Ortalama tahmin: {pred_recovery.mean():.2%}")


LGD Modeli (recovery_rate tahmini) — Test R²: -0.013
Ortalama gerçek recovery rate: 67.15% | Ortalama tahmin: 65.77%


## 6. EAD Modeli — Credit Conversion Factor (CCF)

EAD'i doğrudan modellemek yerine, kalan bakiye oranını (CCF) tahmin ediyoruz: `EAD = financed_amount × CCF`. CCF, kredinin ne kadar erken aşamada temerrüde düştüğüne bağlı.

In [14]:

df_def["ccf"] = df_def["outstanding_at_default_tl"] / df_def["financed_amount_tl"]

X_ead = pd.get_dummies(df_def[["term_months","months_survived","vehicle_segment"]],
                        columns=["vehicle_segment"], drop_first=True)
y_ead = df_def["ccf"]

Xe_train, Xe_test, ye_train, ye_test = train_test_split(X_ead, y_ead, test_size=0.25, random_state=42)
ead_model = LinearRegression()
ead_model.fit(Xe_train, ye_train)

pred_ccf = ead_model.predict(Xe_test)
print(f"EAD/CCF Modeli — Test R²: {r2_score(ye_test, pred_ccf):.3f}")


EAD/CCF Modeli — Test R²: 0.911


## 7. Portföy Bazında Expected Loss (EL = PD × LGD × EAD)

Tüm portföy (test seti + train) için PD tahminlerini üretip, ortalama LGD ve EAD varsayımlarıyla (segment bazında) toplam beklenen zararı hesaplıyoruz.

In [15]:

df_all = df.copy()
X_all = pd.get_dummies(df_all[features_num + features_cat], columns=features_cat, drop_first=True)
X_all = X_all.reindex(columns=X.columns, fill_value=0)  # kolon hizalama

df_all["pd_hat"] = pd_model.predict_proba(X_all)[:,1]

# Segment bazlı ortalama LGD (gözlemlenen default'lardan; gerçek hayatta segment bazlı LGD modeli kullanılır)
seg_lgd = df_def.groupby(["brand_tier","vehicle_is_used"])["lgd"].mean()
df_all["lgd_hat"] = df_all.apply(lambda r: seg_lgd.get((r["brand_tier"], r["vehicle_is_used"]), df_def["lgd"].mean()), axis=1)

# EAD: financed_amount × ortalama CCF (basitleştirilmiş; production'da ead_model.predict kullanılır)
avg_ccf = df_def["ccf"].mean()
df_all["ead_hat"] = df_all["financed_amount_tl"] * avg_ccf

df_all["expected_loss_tl"] = df_all["pd_hat"] * df_all["lgd_hat"] * df_all["ead_hat"]

total_el = df_all["expected_loss_tl"].sum()
total_exposure = df_all["financed_amount_tl"].sum()
print(f"Toplam finanse edilen tutar: {total_exposure:,.0f} TL")
print(f"Toplam Beklenen Zarar (EL): {total_el:,.0f} TL")
print(f"Portföy bazında EL oranı: {total_el/total_exposure:.2%}")

print("\nBölge bazında Expected Loss:")
print(df_all.groupby("region")["expected_loss_tl"].sum().sort_values(ascending=False).round(0))


Toplam finanse edilen tutar: 1,138,428,560 TL
Toplam Beklenen Zarar (EL): 107,005,430 TL
Portföy bazında EL oranı: 9.40%

Bölge bazında Expected Loss:
region
Marmara              31858987.0
İç Anadolu           20572025.0
Akdeniz              13132250.0
Ege                  12034748.0
Güneydoğu Anadolu    11426025.0
Karadeniz             9082119.0
Doğu Anadolu          8899275.0
Name: expected_loss_tl, dtype: float64


## 8. What-If Simülatörü

Tek bir başvuru profili için değişkenleri değiştirip Expected Loss'un nasıl tepki verdiğini gösteren bir fonksiyon. Bu, dashboard'una doğrudan taşıyabileceğin mantık — örneğin 'peşinatı %10 artırırsak bu segmentte EL ne kadar düşer' sorusuna cevap verir.

In [16]:

def what_if(profile: dict, change: dict | None = None):
    """
    profile: başvuru özelliklerini içeren dict (features_num + features_cat alanları)
    change: profile üzerinde değiştirilecek alan(lar) -> {'down_payment_ratio': 0.30}
    """
    p = profile.copy()
    if change:
        p.update(change)

    row = pd.DataFrame([p])
    row["installment_to_income"] = (
        (row["loan_amount_tl"] * (1 - row["down_payment_ratio"]) * 1.12 / row["term_months"])
        / row["customer_income_tl"]
    )

    X_row = pd.get_dummies(row[features_num + features_cat], columns=features_cat, drop_first=True)
    X_row = X_row.reindex(columns=X.columns, fill_value=0)

    pd_hat = pd_model.predict_proba(X_row)[:,1][0]
    lgd_hat = seg_lgd.get((p["brand_tier"], p["vehicle_is_used"]), df_def["lgd"].mean())
    financed = p["loan_amount_tl"] * (1 - p["down_payment_ratio"])
    ead_hat = financed * avg_ccf
    el = pd_hat * lgd_hat * ead_hat
    return {"pd": pd_hat, "lgd": lgd_hat, "ead": ead_hat, "expected_loss_tl": el, "financed_amount_tl": financed}

base_profile = {
    "findeks_score": 1250, "down_payment_ratio": 0.15, "existing_loan_count": 2,
    "customer_age": 34, "employment_type": "Esnaf/KOBİ", "vehicle_segment": "SUV",
    "brand_tier": "İthal - Ekonomik", "region": "Marmara",
    "loan_amount_tl": 850000, "term_months": 48,
    "customer_income_tl": 45000, "vehicle_is_used": 1,
}

base = what_if(base_profile)
scenario_a = what_if(base_profile, {"down_payment_ratio": 0.35})            # peşinat artışı
scenario_b = what_if(base_profile, {"findeks_score": 1600})                  # daha iyi skor
scenario_c = what_if(base_profile, {"down_payment_ratio": 0.35, "findeks_score": 1600})  # ikisi birden

print("Senaryo karşılaştırması (aynı başvuru, farklı koşullar):\n")
for name, s in [("Baz senaryo", base),
                ("Peşinat %15 -> %35", scenario_a),
                ("Findeks 1250 -> 1600", scenario_b),
                ("Her ikisi birden", scenario_c)]:
    print(f"{name:28s} | PD: {s['pd']:.2%} | LGD: {s['lgd']:.2%} | EAD: {s['ead']:>10,.0f} TL | EL: {s['expected_loss_tl']:>10,.0f} TL")


Senaryo karşılaştırması (aynı başvuru, farklı koşullar):

Baz senaryo                  | PD: 75.76% | LGD: 36.38% | EAD:    444,088 TL | EL:    122,404 TL
Peşinat %15 -> %35           | PD: 64.85% | LGD: 36.38% | EAD:    339,596 TL | EL:     80,120 TL
Findeks 1250 -> 1600         | PD: 49.07% | LGD: 36.38% | EAD:    444,088 TL | EL:     79,281 TL
Her ikisi birden             | PD: 36.25% | LGD: 36.38% | EAD:    339,596 TL | EL:     44,789 TL


## 9. Sonuç ve Production'a Taşıma Notları

- Bu notebook **metodolojiyi** gösteriyor: PD (lojistik regresyon) → LGD (segment bazlı / regresyon) → EAD (CCF) → EL.
- Production'a taşırken: (1) gerçek WoE/IV binning ile scorecard kurulmalı, (2) Findeks/KKB skoru gerçek API'den çekilmeli, (3) LGD segmentasyonu gerçek tahsilat/icra verisiyle kalibre edilmeli, (4) PSI ile model izleme eklenmeli, (5) `what_if()` fonksiyonu FastAPI endpoint'ine sarılıp React dashboard'undan çağrılabilir.
- Bu yapı, renewal opportunity score ve NPL/FPD risk dashboard'un için doğrudan iskelet olarak kullanılabilir.